# Phase 2 kickoff — failure audit of the baselines

**Runtime: plain CPU** (Runtime → Change runtime type → CPU). This costs **zero compute units** — we're only re-running tests, no model involved.

**What this does:**
1. Re-grades every baseline attempt with our own scorer (independent check of the official 17.59% / 29.94%)
2. Computes exact per-problem results + proper 95% confidence intervals using the repo's tested tools
3. Sorts failures into species (wrong-logic / stutter / broken-syntax) and saves 30 failed answers to Drive for reading

Run once with `RUN_NAME = 'qwen_base'`, then again with `'llama_base'`.

In [ ]:
RUN_NAME = 'qwen_base'   # second pass: 'llama_base'
OFFICIAL_PASS1 = {'qwen_base': 0.1759, 'llama_base': 0.2994}[RUN_NAME]

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, json, os
PHASE1 = '/content/drive/MyDrive/rl-post-training/phase1'
gens_path = glob.glob(f'{PHASE1}/gens_{RUN_NAME}*.json')[0]
gens = json.load(open(gens_path))
print('Loaded', len(gens), 'problems x', len(gens[0]), 'attempts from', gens_path)

In [ ]:
# Our own measuring tools + the exam problems (for their tests)
!git clone -q https://github.com/nidhi1603/post-training-lab.git /content/ptl
!pip install -q datasets
import sys
sys.path.insert(0, '/content/ptl/eval')
from datasets import load_dataset
ds = load_dataset('bigcode/humanevalpack', 'python', split='test', trust_remote_code=True)
assert len(ds) == len(gens)
print('Exam loaded:', len(ds), 'problems')

In [ ]:
# Re-grade every attempt by RUNNING ITS TESTS (subprocess + timeout).
# ~10-25 min on the free CPU. Progress prints every 20 problems.
import subprocess, sys as _sys, tempfile
from concurrent.futures import ThreadPoolExecutor

def run_one(candidate, test, entry_point, timeout=8):
    prog = candidate + '\n\n' + test + f'\n\ncheck({entry_point})\n'
    with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False) as f:
        f.write(prog); path = f.name
    try:
        r = subprocess.run([_sys.executable, path], capture_output=True, timeout=timeout)
        return r.returncode == 0
    except subprocess.TimeoutExpired:
        return False
    finally:
        os.unlink(path)

results = []
with ThreadPoolExecutor(max_workers=4) as pool:
    for i, row in enumerate(ds):
        flags = list(pool.map(lambda g: run_one(g, row['test'], row['entry_point']), gens[i]))
        results.append({'task_id': row['task_id'], 'n': len(flags), 'c': sum(flags),
                        'flags': flags})
        if (i + 1) % 20 == 0:
            print(f'{i+1}/{len(ds)} problems graded')
print('Done.')

In [ ]:
# Independent score + exact confidence interval, using the repo's tested tools
from pass_at_k import aggregate_pass_at_k, mean_and_ci
mean, per_problem = aggregate_pass_at_k([(r['n'], r['c']) for r in results], k=1)
ci = mean_and_ci(per_problem)
print(f"Our scorer:      pass@1 = {mean*100:.2f}%  [95% CI: {ci['ci_low']*100:.2f} - {ci['ci_high']*100:.2f}]")
print(f"Official number: pass@1 = {OFFICIAL_PASS1*100:.2f}%")
print(f"Agreement gap:   {abs(mean-OFFICIAL_PASS1)*100:.2f} points  (small = two independent graders agree)")

In [ ]:
# Failure species: wrong-logic vs stutter vs broken-syntax
from collections import Counter

def species(gen):
    try:
        compile(gen, '<gen>', 'exec')
    except SyntaxError:
        return 'broken_syntax'
    lines = [l for l in gen.splitlines() if l.strip()]
    if lines and Counter(lines).most_common(1)[0][1] >= 5:
        return 'stutter'
    return 'wrong_logic'

failed_attempts = Counter()
for i, r in enumerate(results):
    for j, ok in enumerate(r['flags']):
        if not ok:
            failed_attempts[species(gens[i][j])] += 1
total_failed = sum(failed_attempts.values())
print(f'Failed attempts: {total_failed}')
for sp, cnt in failed_attempts.most_common():
    print(f'  {sp:<15} {cnt:>5}  ({cnt/total_failed*100:.1f}%)')
unsolved = [r for r in results if r['c'] == 0]
print(f"\nProblems the model NEVER solved in 20 tries: {len(unsolved)}/{len(results)}")

In [ ]:
# Save: per-problem results (the repo's contract) + 30 failures for reading
slim = [{'task_id': r['task_id'], 'n': r['n'], 'c': r['c']} for r in results]
res_path = f'{PHASE1}/results_{RUN_NAME}.json'
json.dump(slim, open(res_path, 'w'), indent=1)

lines = []
for r in unsolved[:30]:
    i = next(k for k, rr in enumerate(results) if rr['task_id'] == r['task_id'])
    g = gens[i][0]
    lines.append(f"{'='*70}\n{r['task_id']}  (0/{r['n']} passed)  species: {species(g)}\n{'='*70}\n{g[:3000]}\n")
fail_path = f'{PHASE1}/failures_{RUN_NAME}.txt'
open(fail_path, 'w').write('\n'.join(lines))
print('Saved:', res_path)
print('Saved:', fail_path, f'({len(unsolved[:30])} never-solved problems)')
print('\nNow flip RUN_NAME to the other model at the top and Run all again.')